# Why Britain voted Leave: data cleaning

**Project:** Why Britain voted Leave, and where the explanation breaks

**Goal of this notebook:** turn the raw files into clean tables, and record every change, so anyone can rerun the work from the start.

| Raw file | What it holds | Clean table |
|---|---|---|
| `EU-referendum-results-and-characteristics-data.xlsx` | Results and area characteristics | `gb_areas_clean.csv` |
| `EU-referendum-result-data.xls` | Second copy of the results, used as a check | Used as a check |
| `brexit_gb_clean.csv` | Our earlier merged table, used as a check | Replaced by `gb_areas_clean.csv` |
| `RP14-50.pdf` | Commons Library 2014 results by council | `scotland_2014_2016.csv` |
| `DS_10283_2614.zip` | Edinburgh DataShare 2014 results, used as a check | Used as a check |
| `Scottish-independence-referendum-report.pdf` | Electoral Commission report, used as a check | Used as a check |
| `uk_election_data_v2.csv` | 2015 constituency results with Leave estimates | `constituencies_clean.csv` |
| `Brexit_tweets_stance.zip` | Tweet IDs and stance labels | `tweets_hourly_stance.csv`, `tweets_daily_stance.csv` |

The notebook also writes `cleaning_log.csv`, which lists each action.

**Before running:**

1. Install the extra libraries once, in a terminal: `pip install pandas numpy openpyxl xlrd pypdf`
2. Keep all the raw files above together in one folder inside Documents, Desktop or Downloads. Section 1 finds that folder by itself.
3. Use **Run All**. The tweet step takes the longest because it reads over 1,000 small files.

**Cleaning rules:** we never edit the raw files. We recalculate shares from vote counts, and we leave real gaps blank instead of guessing.

## 1. Setup

We import the libraries, find the two folders and create an empty log. The `note()` function adds one line to that log each time we change something.

**What this cell does**

- It imports the tools for files, patterns, zip archives, tables and PDFs.
- `find_folder()` searches your usual folders for a file and returns the folder that holds it. This saves us from typing long paths.
- `note()` adds one line to the cleaning log.

**Why it matters:** the log gives us a record of every change, so anyone can see what we did to the data and why.

In [1]:
# Standard Python tools for files, patterns and zip archives
import glob
import io
import os
import re
import zipfile
from pathlib import Path

# pandas handles tables and numpy handles the maths
import numpy as np
import pandas as pd

# pypdf reads text from PDF files
from pypdf import PdfReader

def find_folder(file_name):
    """Search the usual places and return the folder that holds this file."""
    places = [Path.cwd(), Path.cwd().parent, Path.home() / "Documents", Path.home() / "Desktop", Path.home() / "Downloads"]
    for place in places:
        # os.walk looks through a folder and everything inside it
        for folder, subfolders, files in os.walk(place):
            if file_name in files:
                return Path(folder)
    raise FileNotFoundError(f"Could not find {file_name} in Documents, Desktop or Downloads")

# Find the raw files, so we don't have to type the folder path
RAW_DIR = find_folder("EU-referendum-results-and-characteristics-data.xlsx")

# Save to the existing clean folder, or create one next to the raw folder
try:
    CLEAN_DIR = find_folder("gb_areas_clean.csv")
except FileNotFoundError:
    CLEAN_DIR = RAW_DIR.parent / "Clean data"

print("Raw folder:", RAW_DIR)
print("Clean folder:", CLEAN_DIR)

# Create the clean folder if it doesn't exist yet
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

# The cleaning log starts empty
log = []

def note(table, item, rows_affected, action):
    """Record one cleaning action."""
    log.append({"table": table, "item": item, "rows_affected": rows_affected, "action": action})

### Check that every raw file is there

It is better to find a missing file now than halfway through the notebook.

**What this cell does:** it lists the raw files and stops with a clear message if any are missing.

`brexit_gb_clean.csv` isn't on this list, because the notebook can run without it.

In [2]:
# Every file this notebook needs
required = [
    "EU-referendum-results-and-characteristics-data.xlsx",
    "EU-referendum-result-data.xls",
    "uk_election_data_v2.csv",
    "RP14-50.pdf",
    "Scottish-independence-referendum-report.pdf",
    "DS_10283_2614.zip",
    "Brexit_tweets_stance.zip",
]

# List any file that can't be found
missing_files = [name for name in required if not (RAW_DIR / name).exists()]

# Stop with a clear message if something is missing
if missing_files:
    raise FileNotFoundError(f"Missing from RAW_DIR: {missing_files}")
print("All", len(required), "raw files found")

All 7 raw files found


## 2. Referendum results

We have two copies of the results, so we can check one against the other. We also compare the totals with the official UK result: 17,410,742 Leave and 16,141,241 Remain.

**What this cell does:** `pd.ExcelFile()` opens the workbook, so we can read each sheet from it. `book.sheet_names` lists them. We then load the results from both files and compare their sizes. Both should have 382 rows.

In [3]:
# Open the workbook once, because we read many sheets from it
book = pd.ExcelFile(RAW_DIR / "EU-referendum-results-and-characteristics-data.xlsx")
print("Sheets:", book.sheet_names)

# Load both copies of the results
results = book.parse("Results")
results_copy = pd.read_excel(RAW_DIR / "EU-referendum-result-data.xls", sheet_name="Full UK data")
print("Rows in each file:", len(results), len(results_copy))

Sheets: ['Results', 'Social grade', 'EP 2014', 'Age', 'Immigration', 'Highest qualification', 'Tax credits', 'Child poverty', 'Median pay', 'JSA', 'Deprivation']
Rows in each file: 382 382


**What this cell does**

- `.reindex()` puts the second file in the same order as the first, because comparing unordered rows would give false differences.
- The loop compares six vote count columns and counts the differences.
- It prints the first `Pct_Leave` value from each file, to show the scale problem.
- It adds up the UK totals to compare with the official result.

**How to read the result:** 0 differences means the counts agree exactly. The `Pct_Leave` values show the same number on two different scales.

In [4]:
# Line up the second copy so its rows match the first
copy_aligned = results_copy.set_index("Area_Code").reindex(results["Area_Code"]).reset_index()

# The vote counts must be identical
for column in ["Electorate", "Votes_Cast", "Valid_Votes", "Remain", "Leave", "Rejected_Ballots"]:
    differences = (results[column].values != copy_aligned[column].values).sum()
    print(f"{column}: {differences} differences")

# The percentage columns use different scales, so we can't trust them
print("Pct_Leave, first file:", round(results["Pct_Leave"].iloc[0], 3))
print("Pct_Leave, second file:", round(copy_aligned["Pct_Leave"].iloc[0], 3))

# Compare the totals with the official result
print("Leave total:", f"{results['Leave'].sum():,}")
print("Remain total:", f"{results['Remain'].sum():,}")
print(f"Leave share: {100 * results['Leave'].sum() / results['Valid_Votes'].sum():.1f}%")

note("gb_areas", "two results files", len(results),
     "vote counts identical; the Pct_ columns are proportions in one file and percentages in the other, so all shares are recalculated from counts")

Electorate: 0 differences
Votes_Cast: 0 differences
Valid_Votes: 0 differences
Remain: 0 differences
Leave: 0 differences
Rejected_Ballots: 0 differences
Pct_Leave, first file: 0.609
Pct_Leave, second file: 60.89
Leave total: 17,410,742
Remain total: 16,141,241
Leave share: 51.9%


**Problem found:** the vote counts agree, but the `Pct_` columns are on different scales. We recalculate every share from the vote counts.

Northern Ireland and Gibraltar have no matching rows in the other sheets, so we remove them. That leaves 380 areas in Great Britain.

**What this cell does**

- `~` means "not", so this keeps every area except Northern Ireland and Gibraltar.
- It makes a smaller table with clearer column names.
- It recalculates `leave_pct`, `remain_pct` and `turnout_pct` from the vote counts.
- The first letter of each code gives the nation: E for England, S for Scotland and W for Wales.

**Why it matters:** counts are the most reliable numbers in the file, so every share now uses one scale and one method.

In [5]:
# Remove the two areas that have no comparable data
gb = results[~results["Area_Code"].isin(["N92000002", "GI"])]
note("gb_areas", "Northern Ireland, Gibraltar", len(results) - len(gb),
     "removed: census and other sheets have no comparable rows")

# Keep the columns we need, with clearer names
areas = pd.DataFrame({
    "code": gb["Area_Code"],
    "area": gb["Area"],
    "region": gb["Region"],
    "electorate": gb["Electorate"],
    "votes_cast": gb["Votes_Cast"],
    "valid_votes": gb["Valid_Votes"],
    "remain_votes": gb["Remain"],
    "leave_votes": gb["Leave"],
})

# Recalculate the shares from the vote counts
areas["leave_pct"] = 100 * areas["leave_votes"] / areas["valid_votes"]
areas["remain_pct"] = 100 * areas["remain_votes"] / areas["valid_votes"]
areas["turnout_pct"] = 100 * areas["votes_cast"] / areas["electorate"]

# The first letter of an area code shows its nation
areas["nation"] = np.select([areas["code"].str[0] == "S", areas["code"].str[0] == "W"], ["Scotland", "Wales"], "England")

print(areas.shape)
print(areas["nation"].value_counts())

(380, 12)
nation
England     326
Scotland     32
Wales        22
Name: count, dtype: int64


## 3. Area characteristics

Each sheet has its own layout, with notes above and below the data. Every data row does contain an area code, though, so this function finds those rows and ignores the rest.

**What this cell does**

- `read_sheet()` reads a sheet without assuming where its table starts. A regular expression, `[ESWKN]\d{8}`, finds cells that look like an area code: one letter followed by 8 digits.
- It keeps those rows and removes repeated codes.
- `to_number()` turns values into numbers. Symbols such as `#` or `-` become blank, because they mean the value isn't available.

**Why it matters:** we can reuse these two functions on every sheet, which avoids repeating code.

In [6]:
def read_sheet(name):
    """Read one sheet and keep only rows that have an area code."""
    sheet = book.parse(name, header=None)

    def find_code(row):
        # The code is always in one of the first three columns
        for value in row.iloc[:3]:
            text = str(value).strip()
            if re.fullmatch(r"[ESWKN]\d{8}", text):
                return text
        return None

    # Keep the rows with a code, and only the first row for each code
    codes = sheet.apply(find_code, axis=1)
    keep = codes.notna() & ~codes.duplicated()
    return sheet[keep].reset_index(drop=True), codes[keep].reset_index(drop=True)

def to_number(column):
    """Convert a column to numbers. Text such as '#' or '-' becomes blank."""
    return pd.to_numeric(column, errors="coerce")

### Census 2011: education, social grade and age

We turn counts into percentages, so large and small areas can be compared.

- **Education:** share of residents aged 16+ with a degree-level qualification (Level 4), and share with no qualifications.
- **Social grade:** share of households in grades AB and DE.
- **Age:** share aged 65+ and share aged 18 to 29. The sheet has one column per year of age, so we add those columns up.

**What this cell does:** for each measure, it reads the sheet, finds the columns it needs and divides by the total to get a percentage.

**Why it matters:** raw counts would mostly show population size. Birmingham would have more graduates than a small town simply because it has more people.

In [7]:
# Education: find the columns by their header names
sheet, codes = read_sheet("Highest qualification")
header = book.parse("Highest qualification", header=None).iloc[7].astype(str)
level4 = header[header.str.contains("Level 4")].index[0]
no_quals = header[header.str.contains("No qualifications")].index[0]
education = pd.DataFrame({
    "code": codes,
    "pct_degree": 100 * to_number(sheet[level4]) / to_number(sheet[2]),
    "pct_noqual": 100 * to_number(sheet[no_quals]) / to_number(sheet[2]),
})

# Social grade: column 2 is the total, 3 is AB and 6 is DE
sheet, codes = read_sheet("Social grade")
grade = pd.DataFrame({
    "code": codes,
    "pct_ab": 100 * to_number(sheet[3]) / to_number(sheet[2]),
    "pct_de": 100 * to_number(sheet[6]) / to_number(sheet[2]),
})

# Age: match each header such as "Age 42" to its column
sheet, codes = read_sheet("Age")
header = book.parse("Age", header=None).iloc[7].astype(str)
age_columns = {}
for column, label in header.items():
    match = re.fullmatch(r"Age (\d+)(?: and over)?", label.strip())
    if match:
        age_columns[int(match[1])] = column

# Add up the single-year columns for each age group
population = to_number(sheet[2])
over_65 = sum(to_number(sheet[age_columns[a]]) for a in age_columns if a >= 65)
age_18_29 = sum(to_number(sheet[age_columns[a]]) for a in age_columns if 18 <= a <= 29)
age = pd.DataFrame({"code": codes, "pct_65plus": 100 * over_65 / population, "pct_18_29": 100 * age_18_29 / population})

print(education.shape, grade.shape, age.shape)

(384, 3) (384, 3) (384, 3)


### Economy: pay, claimant rate and child poverty

**Problem found:** the child poverty sheet still uses 8 older area codes, from before some codes changed. Without a fix, areas such as Glasgow would lose their value when we merge. We replace each old code with the code used in the results.

**What this cell does**

- It reads pay and claimant rate. The claimant rate is multiplied by 100, so all percentages use the same scale.
- `old_codes` is a dictionary that matches each old area code to its current one.
- `.replace()` applies those matches before we merge.

In [8]:
# Median weekly pay
sheet, codes = read_sheet("Median pay")
pay = pd.DataFrame({"code": codes, "median_weekly_pay": to_number(sheet[2])})

# Claimant rate: the sheet gives a proportion, so we multiply by 100
sheet, codes = read_sheet("JSA")
claimant = pd.DataFrame({"code": codes, "claimant_rate": 100 * to_number(sheet[3])})

# Old area code -> current area code
old_codes = {
    "E06000048": "E06000057",  # Northumberland
    "E08000020": "E08000037",  # Gateshead
    "S12000009": "S12000045",  # East Dunbartonshire
    "S12000043": "S12000046",  # Glasgow City
    "E07000097": "E07000242",  # East Hertfordshire
    "E07000100": "E07000240",  # St Albans
    "E07000101": "E07000243",  # Stevenage
    "E07000104": "E07000241",  # Welwyn Hatfield
}

# Child poverty: recode the old codes, then convert to a percentage
sheet, codes = read_sheet("Child poverty")
poverty = pd.DataFrame({"code": codes.replace(old_codes), "child_poverty_pct": 100 * to_number(sheet[4])})

note("gb_areas", "child poverty: pre-2013/14 area codes", len(old_codes),
     "recoded old codes to the codes used in Results (Northumberland, Gateshead, East Dunbartonshire, Glasgow, 4 Hertfordshire districts)")
print("Old codes fixed:", codes.isin(old_codes).sum())

Old codes fixed: 8


### European Parliament election 2014

**Problem found:** Southend's row has its name in the code column, so it wouldn't match the results. We put the correct code back. We also keep only the 382 area rows, because the rows below them are regional totals.

**What this cell does**

- It removes hidden spaces from the column names. The column was called `Code ` with a space, which would break later steps.
- `iloc[:382]` keeps the area rows only.
- It shows the broken Southend row, then writes in its correct code.
- It calculates UKIP's share of valid votes, and turnout.

In [9]:
# Load the sheet and remove stray spaces from the column names
ep = book.parse("EP 2014")
ep.columns = [str(column).strip() for column in ep.columns]

# Keep the area rows and drop the regional totals underneath
ep = ep.iloc[:382].copy()

# Show the broken row
broken = ep["Code"].astype(str) == "Southend-on-Sea"
display(ep.loc[broken, ["Code", "Counting Area", "Region"]])

# Put the correct code in
ep.loc[broken, "Code"] = "E06000033"
note("gb_areas", "EP 2014: Southend row", int(broken.sum()),
     "row had the name in the code column; code restored so UKIP share and turnout both load")

# UKIP share of valid votes, and turnout
european = pd.DataFrame({
    "code": ep["Code"],
    "ukip_2014_pct": 100 * to_number(ep["UKIP"]) / to_number(ep["Valid votes counted"]),
    "ep_2014_turnout_pct": 100 * to_number(ep["Valid votes counted"]) / to_number(ep["Eligible Electorate"]),
})

,Code,Counting Area,Region
177,Southend-on-Sea,Southend,East


### Immigration and deprivation

These two sources cover only part of Great Britain. We keep them, but we can't use them in a Great Britain model:

- **Non-UK born:** this survey estimate has no Scottish councils and is missing 8 small English districts. Column 2 is the total population, column 6 is non-UK born and column 7 is its margin of error.
- **Deprivation:** this index covers England only.

**What this cell does:** it creates the non-UK born share and keeps the survey's margin of error, then reads the English deprivation score. These figures come from a survey sample, so the margin of error shows how uncertain they are in small areas.

In [10]:
# Non-UK born share, plus the survey's margin of error
sheet, codes = read_sheet("Immigration")
immigration = pd.DataFrame({
    "code": codes,
    "pct_nonuk_born": 100 * to_number(sheet[6]) / to_number(sheet[2]),
    "nonuk_ci_thousands": to_number(sheet[7]),
})

# Deprivation score for English districts
sheet, codes = read_sheet("Deprivation")
deprivation = pd.DataFrame({"code": codes, "imd_score_england": to_number(sheet[4])})

print("Immigration rows:", len(immigration))
print("Deprivation rows:", len(deprivation))

Immigration rows: 343
Deprivation rows: 326


## 4. Merge the area table

We join each table on the area code. `validate="1:1"` makes pandas stop with an error if a code appears twice, which protects us from duplicated rows.

**What this cell does**

- The loop joins each table to `areas` using the area code.
- `how="left"` keeps all 380 areas, even if a table has no value for one of them.
- `assert` stops the notebook if we don't have exactly 380 unique areas.
- It then shows which columns have gaps, and where they are.

**How to read the result:** the gaps match the limits we expected in the sources: Scotland is missing from one table, Scotland and Wales from another, and Isles of Scilly has suppressed values.

In [11]:
# Add each table to the main table, one at a time
for table in [education, grade, age, pay, claimant, poverty, european, immigration, deprivation]:
    areas = areas.merge(table, on="code", how="left", validate="1:1")

# We must still have 380 unique areas
assert len(areas) == 380
assert areas["code"].is_unique

# Show the columns that still have gaps
gaps = areas.isna().sum()
display(gaps[gaps > 0].sort_values(ascending=False).to_frame("missing"))

# Which areas are missing from the non-UK born data?
print(areas.loc[areas["pct_nonuk_born"].isna(), "region"].value_counts())

# Which area has no pay or child poverty value?
print(areas.loc[areas["median_weekly_pay"].isna(), "area"].tolist())
print(areas.loc[areas["child_poverty_pct"].isna(), "area"].tolist())

,missing
imd_score_england,54
nonuk_ci_thousands,40
pct_nonuk_born,40
median_weekly_pay,1
child_poverty_pct,1


region
Scotland                    32
East Midlands                2
South West                   2
London                       1
North West                   1
West Midlands                1
Yorkshire and The Humber     1
Name: count, dtype: int64
['Isles of Scilly']
['Isles of Scilly']


In [12]:
# Record the gaps we are leaving blank on purpose
note("gb_areas", "pct_nonuk_born", int(areas["pct_nonuk_born"].isna().sum()),
     "left blank, not imputed: the source has no Scottish councils and 8 small English districts. Exclude from GB-wide models")
note("gb_areas", "imd_score_england", int(areas["imd_score_england"].isna().sum()),
     "left blank, not imputed: English index only. Use for England-only analysis")
note("gb_areas", "median_weekly_pay, child_poverty_pct", 1,
     "Isles of Scilly values are suppressed in the source; left blank")

**Why we don't fill the gaps:** all of Scotland is missing from one column. Filling it with an average would invent data for the very region we want to test.

### Compare with our earlier table

`brexit_gb_clean.csv` was our first merged table. We compare it with the new one to confirm the numbers agree, and to see what the fixes added. This step runs only if the file is in `RAW_DIR`.

The earlier table also contains `pred` and `res`, which are outputs from a model, and `leave`, a copy of the Leave share. We don't carry them over, because model outputs must never sit in the input table.

**How to read the result:** `values_changed` counts values that differ between the two tables, and `values_restored` counts values that only the new table has. We expect no changes, and two restored values: Glasgow's child poverty and Southend's European election turnout.

In [13]:
earlier_path = RAW_DIR / "brexit_gb_clean.csv"

if earlier_path.exists():
    earlier = pd.read_csv(earlier_path)
    check = areas.merge(earlier, on="code", suffixes=("", "_old"))

    # New column name -> earlier column name
    pairs = {"pct_ab": "pct_AB", "pct_de": "pct_DE", "pct_nonuk_born": "pct_nonuk", "median_weekly_pay": "median_pay",
             "child_poverty_pct": "child_poverty", "imd_score_england": "imd_score", "ukip_2014_pct": "ukip_2014",
             "ep_2014_turnout_pct": "ep_turnout", "pct_degree": "pct_degree_old", "pct_noqual": "pct_noqual_old",
             "pct_65plus": "pct_65plus_old", "pct_18_29": "pct_18_29_old", "claimant_rate": "claimant_rate_old"}

    # Count values that differ, and values that only the new table has
    rows = []
    for new, old in pairs.items():
        changed = int(((check[new] - check[old]).abs() > 0.01).sum())
        restored = int((check[new].notna() & check[old].isna()).sum())
        rows.append({"column": new, "values_changed": changed, "values_restored": restored})
    display(pd.DataFrame(rows))
    note("gb_areas", "leave, pred, res", len(earlier), "dropped from the earlier file: model outputs must not be inputs")
else:
    print("brexit_gb_clean.csv not found, so this check was skipped")

,column,values_changed,values_restored
0,pct_ab,0,0
1,pct_de,0,0
2,pct_nonuk_born,0,0
3,median_weekly_pay,0,0
4,child_poverty_pct,0,1
5,imd_score_england,0,0
6,ukip_2014_pct,0,0
7,ep_2014_turnout_pct,0,1
8,pct_degree,0,0
9,pct_noqual,0,0


## 5. Scotland: 2014 independence referendum

We have three sources for the same results:

- The **Commons Library paper** (`RP14-50.pdf`) is our main source, because it also gives rejected votes and the electorate.
- The **Edinburgh DataShare** file is our first check.
- The **Electoral Commission report** is our second check.

The results are in a table inside the PDF, so we read the text and use a pattern to pull out each council's row.

**What this cell does**

- `PdfReader` opens the paper, and we find the page containing Table 2.1.
- A regular expression describes one table row, and `finditer()` finds every row that fits it.
- We remove commas so that `59,390` becomes the number `59390`.
- We skip the Scotland total, then check for 32 councils.
- We calculate the Yes share of valid votes, and turnout.

**How to read the result:** the totals should be 1,617,989 Yes and 2,001,926 No, which match the official result in the paper.

In [14]:
# Read the Commons Library paper and find the results table
reader = PdfReader(RAW_DIR / "RP14-50.pdf")
table_text = next(page.extract_text() for page in reader.pages
                  if "Table 2.1" in page.extract_text() and "Aberdeenshire" in page.extract_text())

# A row is: name, yes, no, rejected, votes cast, yes %, no %, turnout %, electorate
pattern = re.compile(r"^([A-Z][A-Za-z ,]+?) ([\d,]+) ([\d,]+) (\d+) ([\d,]+) [\d.]+% [\d.]+% [\d.]+% ([\d,]+)\s*$", re.M)

rows = []
for match in pattern.finditer(table_text):
    # Skip the total row and remove commas from the numbers
    if match[1].strip() != "Scotland":
        rows.append([match[1].strip()] + [int(match[i].replace(",", "")) for i in (2, 3, 4, 5, 6)])

indyref = pd.DataFrame(rows, columns=["council", "yes_votes", "no_votes", "rejected_2014", "votes_cast_2014", "electorate_2014"])

# There must be 32 councils and the official national totals
assert len(indyref) == 32
print("Yes total:", f"{indyref['yes_votes'].sum():,}")
print("No total:", f"{indyref['no_votes'].sum():,}")

# Yes share of valid votes, and turnout
indyref["yes_pct"] = 100 * indyref["yes_votes"] / (indyref["yes_votes"] + indyref["no_votes"])
indyref["turnout_2014_pct"] = 100 * indyref["votes_cast_2014"] / indyref["electorate_2014"]

Yes total: 1,617,989
No total: 2,001,926


**What this cell does**

- It opens the DataShare spreadsheet without unpacking any files.
- It compares each council's Yes and No pair with the paper.
- It searches the Electoral Commission report for every vote count.

**How to read the result:** 0 councils differ, and all 32 counts appear in the report. Three separate sources agree, so we can trust these numbers.

In [15]:
# First check: DataShare. Its spreadsheet sits inside a zip inside a zip
with zipfile.ZipFile(RAW_DIR / "DS_10283_2614.zip") as outer:
    with zipfile.ZipFile(io.BytesIO(outer.read("IndyRef2014.zip"))) as inner:
        datashare = pd.read_excel(io.BytesIO(inner.read("ScotVote3.xls")), sheet_name="Sheet1")

# Keep real council rows and compare each Yes and No pair
datashare = datashare[pd.to_numeric(datashare["Yes"], errors="coerce").notna()]
paper_pairs = set(zip(indyref["yes_votes"], indyref["no_votes"]))
datashare_pairs = set(zip(datashare["Yes"], datashare["No"]))
print("DataShare councils that differ:", len(datashare_pairs - paper_pairs))

# Second check: every count should appear in the Electoral Commission report
report = PdfReader(RAW_DIR / "Scottish-independence-referendum-report.pdf")
report_text = " ".join(page.extract_text() for page in report.pages).replace(",", "")
print("Yes counts found in report:", sum(str(v) in report_text for v in indyref["yes_votes"]), "of 32")
print("No counts found in report:", sum(str(v) in report_text for v in indyref["no_votes"]), "of 32")

DataShare councils that differ: 0
Yes counts found in report: 32 of 32
No counts found in report: 32 of 32


**Problem found:** the Commons Library table has council names but no area codes, so we join on the name. Edinburgh is written differently in the two tables, so we rename it to make all 32 councils match.

**What this cell does**

- It renames Edinburgh to match the results table.
- `^` finds names that appear in only one table, so an empty result means all 32 match.
- It joins the 2014 and 2016 data, with `validate="1:1"` protecting against duplicates.
- It keeps the columns the Scotland section needs.

**How to read the result:** all 32 councils voted Remain, and 4 voted Yes.

In [16]:
# Edinburgh is written differently in the paper and in the results table
indyref["area"] = indyref["council"].replace({"Edinburgh, City of": "City of Edinburgh"})

# Check that all 32 names match before joining
scottish_areas = areas[areas["nation"] == "Scotland"]
unmatched = set(indyref["area"]) ^ set(scottish_areas["area"])
print("Names that don't match:", sorted(unmatched))
assert not unmatched

# Join 2014 results with the 2016 results and area measures
scotland = scottish_areas.merge(indyref, on="area", validate="1:1")
scotland = scotland[["code", "area", "leave_pct", "remain_pct", "turnout_pct", "yes_votes", "no_votes", "yes_pct",
                     "turnout_2014_pct", "electorate_2014", "pct_degree", "pct_de", "claimant_rate",
                     "child_poverty_pct", "median_weekly_pay", "ukip_2014_pct", "pct_65plus"]]

# All 32 councils voted Remain, and 4 voted Yes
print("Councils:", len(scotland))
print("Voted Remain:", (scotland["leave_pct"] < 50).sum())
print("Voted Yes:", (scotland["yes_pct"] > 50).sum())

note("scotland", "2014 results", len(scotland),
     "taken from Commons Library RP14/50 table 2.1 and cross-checked with the DataShare file; joined on council name because the paper has no area codes")

Names that don't match: []
Councils: 32
Voted Remain: 32
Voted Yes: 4


## 6. Constituencies

This file has one row per UK constituency. It isn't part of the project plan yet, but we clean it so it's ready if we need it.

- A blank party vote means the party didn't stand, so we fill it with 0.
- Region names differ from our area table, so we make them match.
- The Leave and Remain shares are labelled as estimates, because constituencies had no official counts and the file doesn't name a source.

**What this cell does**

- It fills blank party votes with 0 and makes them whole numbers.
- It checks that the party votes add up to `ValidVotes` in every seat.
- It changes names such as "South East England" to "South East", to match the area table.

In [17]:
# Load the file
seats = pd.read_csv(RAW_DIR / "uk_election_data_v2.csv")
print(seats.shape)

# Blank means the party did not stand, so the votes are 0
parties = ["Con", "Lab", "UKIP", "LD", "SNP", "Grn", "DUP", "PC", "SF", "UUP", "SDLP", "Alliance", "Ind", "Spk", "Others"]
seats[parties] = seats[parties].fillna(0).astype(int)
note("constituencies", "party vote columns", len(seats), "blank means the party did not stand; filled with 0")

# Check that the party votes add up to the valid votes
print("Seats where votes don't add up:", (seats[parties].sum(axis=1) != seats["ValidVotes"]).sum())

# Make the region names match the area table
seats["Region"] = seats["Region"].str.replace(" England", "", regex=False).replace(
    {"East of": "East", "Yorkshire and the Humber": "Yorkshire and The Humber"})

(650, 30)
Seats where votes don't add up: 0


**What this cell does**

- It finds seats where Remain and Leave are more than 0.5 points away from 100. We allow a small gap because the values are rounded.
- It renames columns so that `_est` clearly marks estimates.
- It adds two True or False columns: one for seats that changed party in 2017, and one for the bad rows.

In [18]:
# Remain and Leave should add up to about 100
bad_sum = (seats["RemainPct"] + seats["LeavePct"] - 100).abs() > 0.5
display(seats.loc[bad_sum, ["Constituency", "Region", "RemainPct", "LeavePct"]])

# Use clearer names, and mark the shares as estimates
seats = seats.rename(columns={
    "2017_winner": "winner_2017", "WinningParty": "winner_2015", "RemainPct": "remain_pct_est",
    "LeavePct": "leave_pct_est", "LeaveMajority": "leave_margin_est"})

# Add a flag for seats that changed hands, and for the three bad rows
seats["seat_changed_2017"] = seats["winner_2015"] != seats["winner_2017"]
seats["remain_leave_sum_error"] = bad_sum

note("constituencies", "remain + leave does not equal 100", int(bad_sum.sum()),
     "flagged, not corrected: Brighton Kemptown, Islwyn, Richmond Park. The source does not show which number is wrong")
note("constituencies", "TurnoutPct2015", int(seats["TurnoutPct2015"].isna().sum()),
     "left blank: cannot be rebuilt because the file has no electorate column")

,Constituency,Region,RemainPct,LeavePct
86,Brighton Kemptown,South East,57.6,43.4
302,Islwyn,Wales,45.3,58.8
466,Richmond Park,London,73.3,27.7


**Why we flag these three rows without fixing them:** we can see that the two numbers don't add up, but not which one is wrong.

## 7. Tweets

The archive has one file per hour, and each line holds a tweet ID and a stance label. It holds no tweet text, no user details and no local area.

**Three things to know:**

- **The ReadMe file** sits among the data files. It would load as 23 broken rows, so we skip it.
- **The labels can mislead.** The dataset's ReadMe says Positive means supporting Brexit, so Positive is Leave and Negative is Remain. We rename them to avoid mistakes.
- **Tweet times come from the ID.** A Twitter ID contains its creation time, so we can recover the time from the ID.

**What this cell does**

- It opens the zip file and goes through the files in order.
- A regular expression keeps only names shaped like `2016-05-12_13`. This skips the ReadMe, and the hidden `__MACOSX` files a Mac can add.
- It reads each file as tab-separated data, and `pd.concat()` stacks them into one table.
- `dtype` keeps tweet IDs as whole numbers. They are too long to store safely as decimals.
- `assert` checks that every ID is unique and every row has a label.

In [19]:
# Read every hourly file from the zip, without unpacking it
tables = []
with zipfile.ZipFile(RAW_DIR / "Brexit_tweets_stance.zip") as archive:
    for name in sorted(archive.namelist()):
        # Hourly files look like 2016-05-12_13, which skips the ReadMe
        if re.fullmatch(r"\d{4}-\d\d-\d\d_\d\d", Path(name).name) and "__MACOSX" not in name:
            with archive.open(name) as file:
                tables.append(pd.read_csv(file, sep="\t", header=None, names=["id", "label"], dtype={"id": "int64", "label": "str"}))

tweets = pd.concat(tables, ignore_index=True)
print("Hourly files:", len(tables))
print("Tweets:", f"{len(tweets):,}")

# Every ID must be unique and every row must have a label
assert tweets["id"].is_unique
assert tweets["label"].notna().all()
print(tweets["label"].value_counts())

note("tweets", "ReadMe.txt", 23, "excluded from the load (it would otherwise appear as 23 broken rows)")

Hourly files: 1045
Tweets: 5,499,483
label
Neutral     2280461
Positive    2033643
Negative    1185379
Name: count, dtype: int64


**What this cell does**

- `.map()` renames the labels.
- `>> 22` is a bit shift. It removes the lower part of the ID and leaves the time in milliseconds since Twitter's start date, so we add that date to get the real time.
- `.floor("h")` rounds each time down to the hour.
- It compares the 23 June shares with the published paper.

**How to read the result:** we get 28.0% Leave and 23.9% Remain, close to the paper's 28.5% and 22.4%. That supports the ReadMe's meaning of the labels. A small difference is expected, because the day boundaries may differ.

In [20]:
# Give the labels clear names
tweets["stance"] = tweets["label"].map({"Positive": "leave", "Negative": "remain", "Neutral": "neutral"})
note("tweets", "labels", len(tweets), "Positive=leave, Negative=remain, Neutral=neutral, following the dataset ReadMe")

# Recover the time: the first bits of an ID count milliseconds from Twitter's start date
TWITTER_EPOCH_MS = 1288834974657
tweets["time"] = pd.to_datetime((tweets["id"].to_numpy() >> 22) + TWITTER_EPOCH_MS, unit="ms", utc=True)
tweets["hour"] = tweets["time"].dt.floor("h")

print("First tweet:", tweets["time"].min())
print("Last tweet:", tweets["time"].max())

# Check the labels against the paper, which gives 28.5% Leave and 22.4% Remain on referendum day
day = tweets[tweets["time"].dt.strftime("%Y-%m-%d") == "2016-06-23"]
print((100 * day["stance"].value_counts(normalize=True)).round(1))

First tweet: 2016-05-12 13:58:34.063000+00:00
Last tweet: 2016-06-25 02:59:59.404000+00:00
stance
neutral    48.2
leave      28.0
remain     23.9
Name: proportion, dtype: float64


### Count tweets per hour and per day

We don't save 5.5 million rows, because the analysis needs only the counts. We also mark each hour as one of three periods, using the times polls opened and closed (07:00 and 22:00 UK time, which are 06:00 and 21:00 UTC):

- `pre_vote`: before polls opened
- `polls_open`: while people were voting
- `post_close`: after polls closed

An hour with no file stays blank, because zero would wrongly say that nobody tweeted.

**What this cell does**

- `groupby` with `unstack()` makes one row per hour and one column per stance.
- `date_range` and `reindex` add any missing hour as a blank row.
- `np.select` gives each hour its period.
- `.resample("D").sum()` adds hours up into days.
- `leave_share_decided` is Leave divided by Leave plus Remain, which leaves out neutral tweets.

In [21]:
# Count tweets by hour and stance
hourly = tweets.groupby(["hour", "stance"]).size().unstack(fill_value=0).reindex(columns=["leave", "remain", "neutral"])

# Add any missing hour as a blank row
all_hours = pd.date_range(hourly.index.min(), hourly.index.max(), freq="h", name="hour_utc")
hourly = hourly.reindex(all_hours)
hourly["file_missing"] = hourly["leave"].isna()
print("Missing hours:", hourly.index[hourly["file_missing"]].tolist())

# Mark each hour's period
polls_open = pd.Timestamp("2016-06-23 06:00", tz="UTC")
polls_close = pd.Timestamp("2016-06-23 21:00", tz="UTC")
hourly["period"] = np.select([hourly.index < polls_open, hourly.index < polls_close], ["pre_vote", "polls_open"], "post_close")

# Tweets in each period
print(hourly.groupby("period")[["leave", "remain", "neutral"]].sum().sum(axis=1).astype(int))

# Add the hours up into days
daily = hourly.drop(columns=["file_missing", "period"]).resample("D").sum()
daily.index.name = "date_utc"
daily["total"] = daily.sum(axis=1)

# Leave share of tweets that took a side
daily["leave_share_decided"] = daily["leave"] / (daily["leave"] + daily["remain"])

note("tweets", "file for 2016-06-18 13:00 UTC", int(hourly["file_missing"].sum()), "hour has no file; kept as a blank row, not treated as zero")
note("tweets", "rows after polls closed", int((tweets["time"] >= polls_close).sum()), "kept but flagged post_close; exclude from pre-vote analysis")

Missing hours: [Timestamp('2016-06-18 13:00:00+0000', tz='UTC')]
period
polls_open     390012
post_close    1440802
pre_vote      3668669
dtype: int64


**Open question:** the files hold 5,499,483 tweets, but the ReadMe says 4,508,440. We can't explain the gap yet. Many of the extra tweets may be post-vote reactions, because the running total passes 4.5 million early on 24 June. For "before the vote" analysis, we use only `pre_vote` hours.

## 8. Save the clean tables

We sort the area tables, save every table to `CLEAN_DIR` and show the cleaning log.

**What this cell does:** `.to_csv()` saves each table. `index=False` stops pandas adding a row-number column, and `float_format="%.4f"` rounds decimals to 4 places to keep the files tidy. The tweet tables keep their time index, because it holds the dates.

In [22]:
# Save the tables
areas.sort_values("code").to_csv(CLEAN_DIR / "gb_areas_clean.csv", index=False, float_format="%.4f")
scotland.sort_values("area").to_csv(CLEAN_DIR / "scotland_2014_2016.csv", index=False, float_format="%.4f")
seats.to_csv(CLEAN_DIR / "constituencies_clean.csv", index=False)
hourly.to_csv(CLEAN_DIR / "tweets_hourly_stance.csv")
daily.to_csv(CLEAN_DIR / "tweets_daily_stance.csv")

# Save the log of every cleaning action
cleaning_log = pd.DataFrame(log)
cleaning_log.to_csv(CLEAN_DIR / "cleaning_log.csv", index=False)

# Show each saved table's size
for name, table in [("gb_areas_clean", areas), ("scotland_2014_2016", scotland), ("constituencies_clean", seats),
                    ("tweets_hourly_stance", hourly), ("tweets_daily_stance", daily)]:
    print(f"{name}: {table.shape[0]} rows, {table.shape[1]} columns")

display(cleaning_log[["table", "item", "rows_affected"]])

gb_areas_clean: 380 rows, 26 columns
scotland_2014_2016: 32 rows, 17 columns
constituencies_clean: 650 rows, 32 columns
tweets_hourly_stance: 1046 rows, 5 columns
tweets_daily_stance: 45 rows, 5 columns


,table,item,rows_affected
0,gb_areas,two results files,382
1,gb_areas,"Northern Ireland, Gibraltar",2
2,gb_areas,child poverty: pre-2013/14 area codes,8
3,gb_areas,EP 2014: Southend row,1
4,gb_areas,pct_nonuk_born,40
5,gb_areas,imd_score_england,54
6,gb_areas,"median_weekly_pay, child_poverty_pct",1
7,gb_areas,"leave, pred, res",380
8,scotland,2014 results,32
9,constituencies,party vote columns,650


## 9. Final checks

We reload the saved files and test the facts we rely on. Each `assert` stops the notebook if a check fails.

**What this cell does:** it reopens the saved files and tests them. Each `assert` states something that must be true, and stops the notebook if it isn't.

**Why it matters:** this tests the files we'll really use, not just the tables in memory. If you see `All checks passed`, the clean data is ready for `01_data_exploration.ipynb`.

In [23]:
# Reload what we saved, so we test the real files
check_areas = pd.read_csv(CLEAN_DIR / "gb_areas_clean.csv")
check_scotland = pd.read_csv(CLEAN_DIR / "scotland_2014_2016.csv")

# Area table: right size, unique areas and sensible ranges
assert len(check_areas) == 380 and check_areas["code"].is_unique
assert check_areas["leave_pct"].between(0, 100).all()
assert (check_areas["leave_votes"] + check_areas["remain_votes"] == check_areas["valid_votes"]).all()

# Scotland: official national totals
assert check_scotland["yes_votes"].sum() == 1617989
assert check_scotland["no_votes"].sum() == 2001926

# Tweets: every tweet is counted exactly once
assert hourly[["leave", "remain", "neutral"]].sum().sum() == len(tweets)

# A known result from the feasibility check
r = check_areas["leave_pct"].corr(check_areas["pct_degree"])
print(f"Leave vs graduate share: r = {r:.2f}")
print("All checks passed")

Leave vs graduate share: r = -0.77
All checks passed


## 10. Summary

**Fixes made**

- Recalculated all shares from vote counts, because the two results files use different percentage scales.
- Restored Southend's European election row.
- Recoded 8 old area codes in the child poverty data, which restored Glasgow.
- Dropped model outputs from the input table.
- Renamed the tweet labels to `leave`, `remain` and `neutral`, and skipped the ReadMe file.

**Gaps left blank on purpose**

- `pct_nonuk_born`: 40 areas, including all of Scotland.
- `imd_score_england`: 54 areas, all of Scotland and Wales.
- Isles of Scilly: median pay and child poverty.
- Constituency turnout: 129 seats.

**Still open**

- Why the tweet files hold about 1 million more tweets than the ReadMe states.
- Which number is wrong in the 3 flagged constituencies, and where the file's Leave estimates come from.
- Whether the constituency table has a role in the project.
- The EUvsDisinfo dataset still needs downloading for Part 3.

**Next notebook:** `01_data_exploration.ipynb` reads the tables saved here.